# Task 3 - distance

Needs Task 1 done. The detector is optional: oracle-box evaluation measures the
geometry on its own, so this works even where detection does not.

Two independent estimates from one box, combined by their error models:

| | formula | strong where | weak where |
| --- | --- | --- | --- |
| **D0** known size | `Z = f_y · H_eff / h_px` | any range, no mount needed | floored by how much real object size varies (±17% for cones) |
| **D1** ground plane | `Z = h_c · cos α / sin(θ+α)`, `α = atan((v_b−c_y)/f_y)` | close range, exact for a known mount on flat ground | error grows with `Z²`, so one pixel of box-bottom noise ruins it far away |

`H_eff` is **fitted, not looked up**: the labels are boxes projected from 3D
cuboids, so they are taller than the object, and for barriers they span a merged
row as well. A per-class linear correction `Z = a·Ẑ + b` on the fused estimate
removes what is left. Both are fitted on **train** and scored on **val**.

In [ ]:
# ---- edit this ----
WEIGHTS = "runs/B1/weights/best.pt"   # only needed for the predicted-box pass
DEVICE = ""                           # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from IPython.display import Image, display

from common.notebook import sh

DEVICE_ARGS = ["--device", DEVICE] if DEVICE else []
assert (REPO / "configs/data.yaml").exists(), "run the Task 1 notebook first"
print("repo:", REPO)

In [ ]:
# Fit H_eff and the per-class correction on the train split, into configs/priors.yaml.
sh("-m", "distance.fit_priors")

In [ ]:
# What got fitted, and how far it moved from the physical size.
from common.config import load_yaml

priors = load_yaml("priors.yaml")
print(f"{'class':<11}{'dim':>4}{'H_eff':>8}{'r':>7}{'a':>8}{'b':>8}  ground plane")
for name, p in priors["classes"].items():
    print(
        f"{name:<11}{p.get('dim', 'h'):>4}{p['H_eff']:>8.3f}{p['r']:>7.2f}"
        f"{p.get('a', 1.0):>8.3f}{p.get('b', 0.0):>8.3f}  {p.get('use_ground_plane', False)}"
    )

In [ ]:
# Pass 1: oracle boxes. This is the geometry's own error, with the detector out
# of the picture - so it is measurable for every class, barriers included.
sh("-m", "eval.dist_eval", "--boxes", "oracle")

In [ ]:
# Pass 2: predicted boxes, matched to ground truth at IoU 0.5, true positives
# only. A missed object is a detection failure and is counted in Task 2; mixing
# it in here would hide which half is at fault.
if Path(WEIGHTS).exists():
    sh("-m", "eval.dist_eval", "--boxes", "pred", "--weights", WEIGHTS, *DEVICE_ARGS)
else:
    print(f"{WEIGHTS} not found - oracle numbers above still stand on their own")

In [ ]:
# Side by side. D1 should win at short range and D0 hold up better far out; if
# the correction is doing nothing, a and b were fitted on too few boxes.
import pandas as pd

df = pd.read_csv("results/distance.csv")
display(
    df.pivot_table(
        index=["variant", "class"], columns="stage",
        values=["AbsRel", "within20"], aggfunc="last",
    ).round(3)
)

In [ ]:
# Error against range, per class - the shape matters more than the headline.
import json

import matplotlib.pyplot as plt

detail = json.loads(Path("results/dist_eval_oracle.json").read_text())["per_class"]
fig, axes = plt.subplots(1, len(detail), figsize=(5 * len(detail), 3.6), squeeze=False)
for ax, (name, stages) in zip(axes[0], detail.items()):
    for stage in ("d0", "d1", "corrected"):
        if stage not in stages:
            continue
        bins = stages[stage]["by_range"]
        xs = [b for b, m in bins.items() if m["n"]]
        ys = [bins[b]["AbsRel"] for b in xs]
        ax.plot(xs, ys, marker="o", label=stage)
    ax.axhline(0.15, ls="--", lw=1, color="grey")
    ax.set_title(name)
    ax.set_ylabel("AbsRel")
    ax.legend(fontsize=8)
fig.suptitle("distance error by range, oracle boxes (dashed = 0.15 target)")
fig.tight_layout()
Path("results").mkdir(exist_ok=True)
fig.savefig("results/distance_by_range.png", dpi=120)
plt.show()

## Done when

- cone and barrier **AbsRel ≤ 0.15** with 80%+ within 20% over 0-20 m, on oracle boxes;
- `corrected` is at least as good as `fused`, and `fused` at least as good as the better
  of `d0` / `d1` — if not, the error models driving the weights are wrong;
- the predicted-box numbers are close to the oracle ones. A large gap means box quality,
  not geometry, is the limit.

Stop sign distance has no nuScenes depth, so it is unmeasured here — it needs the
tape-measured photo set at 2, 4, 6, 8 m.

Next: Task 4, export and benchmark; then Task 5 wires this into the realtime loop with
tracking and a Kalman filter on `Z`.